# EExPolytech-2026

Три прогона, обсуждаемые в разделе Results:

| Рисунок | Прогон | Каталог эксперимента | Особенность |
|---|---|---|---|
| Fig. 2 | run 1 | `2026-03-12_15-39-27_neural_controller-v3` | малый $\Delta u_{max} = 20$, $\gamma = 0.99$; режима оценивания нет |
| Fig. 3 | run 2 | `2026-03-26_16-54-41_neural_controller-v3` | увеличенный $\Delta u_{max} = 100$, обычная целевая энтропия ($-\dim a = -1$) |
| Fig. 4 | run 3 | `2026-06-05_14-36-04_neural_controller-v3` | та же конфигурация, что и run 2, но целевая энтропия понижена до $-5$; ось времени обрезана до 180 мин |


## Настройка: палитра, шрифты, каталог вывода

In [ ]:
import matplotlib.pyplot as plt

from nn_laser_stabilizer.analysis.experiments import (
    ExperimentId,
    NeuralControllerExperiment,
)

# Палитра Okabe-Ito
PHOTODETECTOR = "#0072B2"
BAND_ALPHA = 0.18
SETPOINT = "#D55E00"
CONTROL = "#009E73"
HANDOVER = "#000000"
EVAL_FILL = "#999999"
EVAL_ALPHA = 0.15

In [ ]:
COL_W = 3.39 # ширина колонки IEEE A4, дюймы (86 мм)

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"]
})

In [ ]:
from functools import wraps
from pathlib import Path

OUTPUT_DIR = Path("output") / "eexpolytech-2026"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not getattr(plt.savefig, "_auto_wrapped", False):
    _original_savefig = plt.savefig

    @wraps(_original_savefig)
    def _auto_savefig(filename, *args, **kwargs):
        filename = OUTPUT_DIR / Path(filename).name
        return _original_savefig(filename, *args, **kwargs)

    setattr(_auto_savefig, "_auto_wrapped", True)

    plt.savefig = _auto_savefig

## Функция построения рисунка

Верхняя панель (a) — скользящее среднее сигнала фотодетектора с полосой ±СКВО,
нижняя (b) — сырой управляющий сигнал. Легенда собирается из фактически
нарисованных элементов, поэтому у прогона без режима оценивания
соответствующая запись отсутствует сама собой.

In [ ]:
def plot_run(dir_name: str, out_stem: str, xlim: tuple | None = None, window: int = 5000):
    """Построить пару панелей (фотодетектор / управление) для одного прогона.

    dir_name — каталог эксперимента, out_stem — имя файлов в output/ без расширения,
    xlim — ограничение оси времени в минутах, window — окно скользящего среднего в шагах.

    Фигура строится сразу в конечном размере: ширина COL_W соответствует колонке
    шаблона IEEE A4 (86 мм), поэтому в Word рисунок вставляется без масштабирования.
    """
    exp = NeuralControllerExperiment(ExperimentId.from_dir_name(dir_name))
    ia = exp.interaction

    setpoint = float(exp.params.setpoint)

    # Ось времени: global_step пересчитывается в минуты по полной длительности прогона.
    t_min = ia.time.minutes
    experiment_time_min = float(t_min.iloc[-1])
    x_max = float(ia.global_step.max())

    def global_step_to_min(gs: float) -> float:
        return gs / x_max * experiment_time_min

    pid_end = ia.phases.pid.end
    handover_min = global_step_to_min(pid_end.steps) if pid_end is not None else 0.0
    eval_ranges_min = [
        (global_step_to_min(x0), global_step_to_min(x1))
        for x0, x1 in ia.phases.eval.ranges
    ]

    rolling_mean = ia.env.process_variable.rolling(window).mean()
    rolling_std = ia.env.process_variable.rolling(window).std()

    fig, (ax_pv, ax_co) = plt.subplots(
        2,
        1,
        figsize=(COL_W, 2.6),
        sharex=True,
        constrained_layout=True,
        gridspec_kw={"height_ratios": [1, 1]},
    )
    fig.get_layout_engine().set(hspace=0.04, h_pad=0.02, w_pad=0.02)

    # (a) сигнал фотодетектора
    ax_pv.fill_between(
        t_min,
        rolling_mean - rolling_std,
        rolling_mean + rolling_std,
        color=PHOTODETECTOR,
        alpha=BAND_ALPHA,
        linewidth=0,
        label="±std",
        rasterized=True,
    )

    ax_pv.plot(t_min, rolling_mean, color=PHOTODETECTOR, linewidth=1.0)

    ax_pv.axhline(
        y=setpoint,
        linestyle="--",
        linewidth=1.2,
        color=SETPOINT,
        label="Setpoint",
    )

    ax_pv.axvline(
        x=handover_min,
        color=HANDOVER,
        linestyle="--",
        linewidth=1.2,
        label="Transition to NN control",
    )

    for i, (x0, x1) in enumerate(eval_ranges_min):
        ax_pv.axvspan(
            x0,
            x1,
            color=EVAL_FILL,
            alpha=EVAL_ALPHA,
            linewidth=0,
            label="Evaluation Mode" if i == 0 else None,
        )

    # (b) управляющий сигнал
    ax_co.plot(
        t_min,
        ia.env.control_output,
        color=CONTROL,
        linewidth=0.5,
        rasterized=True,
    )

    ax_co.axvline(x=handover_min, color=HANDOVER, linestyle="--", linewidth=1.2)

    for x0, x1 in eval_ranges_min:
        ax_co.axvspan(x0, x1, color=EVAL_FILL, alpha=EVAL_ALPHA, linewidth=0)

    ax_pv.set_ylabel("Photodetector signal\n(ADC counts)")
    ax_co.set_ylabel("Control signal\n(DAC counts)")
    ax_co.set_xlabel("Time (min)")

    fig.align_ylabels([ax_pv, ax_co])

    ax_pv.grid(False)
    ax_co.grid(False)

    ax_pv.text(
        -0.18, 1.10, "(a)",
        transform=ax_pv.transAxes,
        fontsize=8,
        va="bottom",
        ha="center",
    )

    ax_co.text(
        -0.18, 1.02, "(b)",
        transform=ax_co.transAxes,
        fontsize=8,
        va="bottom",
        ha="center",
    )

    handles, labels = ax_pv.get_legend_handles_labels()
    ax_pv.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False,
        fontsize=7,
        handlelength=1.6,
        columnspacing=1.0,
        borderpad=0.2,
        labelspacing=0.3,
    )

    if xlim is not None:
        ax_co.set_xlim(*xlim)

    plt.savefig(f"{out_stem}.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.svg", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.png", bbox_inches="tight", pad_inches=0.01, dpi=300)

    plt.show()
    return exp

## Fig. 2 — run 1: малый максимальный шаг

Режим оценивания в этом прогоне не запускался.

In [ ]:
exp1 = plot_run(
    "2026-03-12_15-39-27_neural_controller-v3",
    out_stem="eexpolytech_fig2_run1",
)

## Fig. 3 — run 2: увеличенный максимальный шаг

Максимальный шаг поднят до 100, добавлен детерминированный режим оценивания (серые полосы). 

In [ ]:
exp2 = plot_run(
    "2026-03-26_16-54-41_neural_controller-v3",
    out_stem="eexpolytech_fig3_run2",
)

## Fig. 4 — run 3: пониженная целевая энтропия

Та же конфигурация, что и в run 2, но целевая энтропия понижена до −5.

In [ ]:
exp3 = plot_run(
    "2026-06-05_14-36-04_neural_controller-v3",
    out_stem="eexpolytech_fig4_run3",
    xlim=(0, 230),
)